# MR DRE (ACL 2026) — baseline trên Kaggle miễn phí

**Đúng mục tiêu:** chạy lại **quy trình đánh giá của bài báo** trước khi nghiên cứu ReviFence, không can thiệp vào cách agent sửa báo cáo. Chỉ chạy **ResearcherBench (65 câu)** bằng agent gốc **LangChain Open Deep Research**, với Ollama và tìm kiếm miễn phí. Kết quả có **đầy đủ Cov., Fa., Gr., Pre., Inc., Brk.** cho những lượt phản hồi có ý nghĩa (Reflect không có Inc.).

### Chạy từ đầu đến cuối

1. Tạo Notebook trên Kaggle, chọn **GPU T4 x2**, **Internet On** ở Settings. Nếu chưa bật được GPU, xác minh tài khoản.
2. Chọn **Save Version → Save & Run All** (hoặc Run All). Notebook tự chứa mã nguồn và bộ dữ liệu, không cần gắn repo GitHub hay nhập API key.
3. Notebook tự chạy **một câu kiểm tra thật** rồi tiến hành 65 câu, ba loại phản hồi: Content1, Format, Reflect. **Các lần chạy có thể vượt quá một phiên Kaggle miễn phí.**
4. Nếu chưa hoàn thành, tải `baseline-results.zip` từ tab Output về, tải ZIP này lên Kaggle thành một Dataset và **Add Input** Dataset đó vào Notebook. Chạy lại cùng Notebook để tiếp tục, không chấm lại câu đã xong.
5. Kết quả trong `output/metrics/*_summary.csv` và `*_per_question.csv`. **Chỉ khi `complete=True` mới đủ sáu chỉ số với 65/65 câu.**

*Để nhanh nhất có một setting đầy đủ sáu chỉ số*, đổi `FEEDBACK='all'` thành `'content'` trong ô cấu hình. Vẫn phải bật `EXTRA_EVAL='all'`. Các setting còn lại có thể dùng lại báo cáo lượt 1 mà không nghiên cứu lại từ đầu.

**Giới hạn khoa học:** đây là baseline tái lập pipeline và metric với mô hình miễn phí (Qwen3 4B), không phải tái lập được số liệu gốc dùng GPT-4.1/5 agent. Chỉ so sánh với ReviFence khi cả hai dùng đúng cùng cấu hình và bộ phản hồi.


In [ ]:
from pathlib import Path
import base64, io, zipfile, hashlib, time
SESSION_START = time.monotonic()
ROOT = Path('/kaggle/working/MRDRE-Baseline-Kaggle-Complete')
ROOT.parent.mkdir(parents=True, exist_ok=True)
PACKAGE = (
    '...'
    'sAKwCDEgAAaA4CAAAA'
)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PACKAGE))) as z:
    assert z.testzip() is None
    z.extractall('/kaggle/working')
data = ROOT / 'data/ResearcherBench.jsonl'
assert hashlib.sha256(data.read_bytes()).hexdigest() == 'e345fe388af84ccb9cd1d92c52a7d409ab0ed1ca87f4279be8d9e8874e100391'
archives = sorted(Path('/kaggle/input').rglob('baseline-results.zip')) if Path('/kaggle/input').exists() else []
if archives:
    with zipfile.ZipFile(archives[-1]) as z:
        for member in z.namelist():
            assert not member.startswith('/') and '..' not in Path(member).parts, member
        z.extractall('/kaggle/working')
    print('Checkpoint loaded:', archives[-1])
else:
    print('Fresh ResearcherBench run: 65 questions')
print('Code:', ROOT)


## 1. Cài mô hình và khởi động agent nghiên cứu


In [ ]:
import sys, subprocess
subprocess.run([sys.executable, str(ROOT / 'kaggle/bootstrap.py')], cwd=ROOT, check=True)


## 2. Chạy baseline đủ ba loại phản hồi và sáu chỉ số


In [ ]:
import sys, subprocess, time
# Baseline gốc: một dataset, ba loại feedback, đầy đủ sáu metric.
N_QUESTIONS = 65
FEEDBACK = 'all'  # all | content | format | reflect
EXTRA_EVAL = 'all'  # GIỮ 'all' nếu muốn đầy đủ Cov/Fa/Gr/Pre/Inc/Brk
SAFE_SESSION_MINUTES = 630  # Dừng sớm để lưu checkpoint trước giới hạn phiên GPU

def execute(count):
    remaining = max(1, int(SAFE_SESSION_MINUTES - (time.monotonic() - SESSION_START) / 60))
    print(f'Run: count={count}, feedback={FEEDBACK}, all_metrics={EXTRA_EVAL}, minutes_left={remaining}', flush=True)
    return subprocess.run([
        sys.executable, str(ROOT / 'kaggle/run.py'),
        '--count', str(count), '--feedback', FEEDBACK,
        '--chunk-size', '5' if count > 1 else '1',
        '--budget-minutes', str(remaining), '--extra-eval', EXTRA_EVAL,
    ], cwd=ROOT, check=False).returncode

# Chạy thật 1 câu, có kiểm tra cả đánh giá, rồi mới chạy toàn bộ.
smoke = execute(1)
if smoke == 0:
    status = execute(N_QUESTIONS)
    print('STATUS:', 'COMPLETE' if status == 0 else f'INCOMPLETE ({status}), checkpoint saved')
else:
    print('SMOKE FAILED:', smoke, '- inspect /kaggle/working/baseline-logs and saved output')


## 3. Kiểm tra kết quả và lấy checkpoint


In [ ]:
from pathlib import Path
from IPython.display import display, FileLink
work = Path('/kaggle/working')
print('SUMMARY (missing scores = not yet evaluated, not zero):')
for f in sorted((work/'output'/'metrics').glob('*_summary.csv')):
    print('\n', f.name, '\n', f.read_text()[:1600])
archive = work/'baseline-results.zip'
if archive.exists():
    display(FileLink(str(archive)))
else:
    print('No checkpoint ZIP yet; inspect logs:', work/'baseline-logs')
